In [17]:
import geemap
geemap.get_ee_token()

{'client_id': '919740042378-b5bm992hcgddbmf50bjvg4ltfd179d1n.apps.googleusercontent.com',
 'client_secret': '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',
 'refresh_token': '1//0gVaEDpvHMV9FCgYIARAAGBASNwF-L9IrFzkNoIAMSkrKVa71NBaF0lWP--Z2zaaYpVnfV9WrqNNZREe5oTd4V9JGzqoD4qOrqLk',
 'scopes': ['https://www.googleapis.com/auth/earthengine',
  'https://www.googleapis.com/auth/devstorage.full_control']}

In [33]:
import ee
service_account = 'onfarmview@hip-rain-278407.iam.gserviceaccount.com'
credentials = ee.ServiceAccountCredentials(service_account, '../hip-rain-278407-e46952e76fd6.json')
ee.Initialize(credentials)


In [32]:
import ee
import json

service_account = "onfarmview@hip-rain-278407.iam.gserviceaccount.com"
private_key = "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"
ee_token = '1//0gVaEDpvHMV9FCgYIARAAGBASNwF-L9IrFzkNoIAMSkrKVa71NBaF0lWP--Z2zaaYpVnfV9WrqNNZREe5oTd4V9JGzqoD4qOrqLk'
# token_dict = json.loads(ee_token, strict=False)


credentials = ee.ServiceAccountCredentials(
    service_account, key_data=private_key
)
ee.Initialize(credentials)#, **kwargs)



ValueError: ('Could not deserialize key data. The data may be in an incorrect format, it may be encrypted with an unsupported algorithm, or it may be an unsupported key type (e.g. EC curves with explicit parameters).', [<OpenSSLError(code=503841036, lib=60, reason=524556, reason_text=unsupported)>])

In [28]:
token_dict

NameError: name 'token_dict' is not defined

(token_name: str = "EARTHENGINE_TOKEN", auth_mode: str = "notebook", service_account: bool = False, auth_args: Any = {}, user_agent_prefix: str = "geemap", **kwargs: Any) -> Any
token_name (str, optional): The name of the Earth Engine token. Defaults to "EARTHENGINE_TOKEN".

Authenticates Earth Engine and initialize an Earth Engine session

Args:
    token_name (str, optional): The name of the Earth Engine token. Defaults to "EARTHENGINE_TOKEN".
    auth_mode (str, optional): The authentication mode, can be one of paste,notebook,gcloud,appdefault. Defaults to "notebook".
    service_account (bool, optional): If True, use a service account. Defaults to False.
    auth_args (dict, optional): Additional authentication parameters for aa.Authenticate(). Defaults to {}.
    user_agent_prefix (str, optional): If set, the prefix (version-less) value used for setting the user-agent string. Defaults to "geemap".
    kwargs (dict, optional): Additional parameters for ee.Initialize(). For example,
        opt_url='https://earthengine-highvolume.googleapis.com' to use the Earth Engine High-Volume platform. Defaults to {}.

In [19]:
geemap.ee_initialize(auth_mode='gcloud')

In [22]:
import ee
ee.Initialize()
ee.Authenticate()

True

In [ ]:
def ee_initialize(
    token_name="EARTHENGINE_TOKEN",
    auth_mode="notebook",
    service_account=False,
    auth_args={},
    user_agent_prefix="geemap",
    **kwargs,
):
    """Authenticates Earth Engine and initialize an Earth Engine session

    Args:
        token_name (str, optional): The name of the Earth Engine token. Defaults to "EARTHENGINE_TOKEN".
        auth_mode (str, optional): The authentication mode, can be one of paste,notebook,gcloud,appdefault. Defaults to "notebook".
        service_account (bool, optional): If True, use a service account. Defaults to False.
        auth_args (dict, optional): Additional authentication parameters for aa.Authenticate(). Defaults to {}.
        user_agent_prefix (str, optional): If set, the prefix (version-less) value used for setting the user-agent string. Defaults to "geemap".
        kwargs (dict, optional): Additional parameters for ee.Initialize(). For example,
            opt_url='https://earthengine-highvolume.googleapis.com' to use the Earth Engine High-Volume platform. Defaults to {}.
    """
    import httplib2
    from .__init__ import __version__

    user_agent = f"{user_agent_prefix}/{__version__}"
    if "http_transport" not in kwargs:
        kwargs["http_transport"] = httplib2.Http()

    auth_args["auth_mode"] = auth_mode

    if ee.data._credentials is None:
        ee_token = os.environ.get(token_name)
        if in_colab_shell():
            from google.colab import userdata

            try:
                ee_token = userdata.get(token_name)
            except Exception:
                pass
        if service_account:
            try:
                credential_file_path = os.path.expanduser(
                    "~/.config/earthengine/private-key.json"
                )

                if os.path.exists(credential_file_path):
                    with open(credential_file_path) as f:
                        token_dict = json.load(f)
                else:
                    token_name = "EARTHENGINE_TOKEN"
                    ee_token = os.environ.get(token_name)
                    token_dict = json.loads(ee_token, strict=False)
                service_account = token_dict["client_email"]
                private_key = token_dict["private_key"]

                credentials = ee.ServiceAccountCredentials(
                    service_account, key_data=private_key
                )
                ee.Initialize(credentials, **kwargs)

            except Exception as e:
                raise Exception(e)

        else:
            try:
                if ee_token is not None:
                    credential_file_path = os.path.expanduser(
                        "~/.config/earthengine/credentials"
                    )
                    if not os.path.exists(credential_file_path):
                        os.makedirs(
                            os.path.dirname(credential_file_path), exist_ok=True
                        )
                        if ee_token.startswith("{") and ee_token.endswith(
                            "}"
                        ):  # deals with token generated by new auth method (earthengine-api>=0.1.304).
                            token_dict = json.loads(ee_token)
                            with open(credential_file_path, "w") as f:
                                f.write(json.dumps(token_dict))
                        else:
                            credential = (
                                '{"refresh_token":"%s"}' % ee_token
                            )  # deals with token generated by old auth method.
                            with open(credential_file_path, "w") as f:
                                f.write(credential)
                # elif in_colab_shell():
                #     if credentials_in_drive() and (not credentials_in_colab()):
                #         copy_credentials_to_colab()
                #     elif not credentials_in_colab:
                #         ee.Authenticate(**auth_args)
                #         if is_drive_mounted() and (not credentials_in_drive()):
                #             copy_credentials_to_drive()
                #     else:
                #         if is_drive_mounted():
                #             copy_credentials_to_drive()

                ee.Initialize(**kwargs)

            except Exception:
                ee.Authenticate(**auth_args)
                ee.Initialize(**kwargs)

    ee.data.setUserAgent(user_agent)